<a href="https://colab.research.google.com/github/Lucaaa31/Anomaly-Segmentation/blob/master/EoMT_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 4: EoMT Evaluation on Cityscapes val

Compare the two EoMT checkpoints on Cityscapes val:

* **EoMT-CS** (Cityscapes-trained, semantic, 19 classes): direct predictions
* **EoMT-COCO** (COCO-trained, panoptic, 133 classes): panoptic predictions + semantic remapping to Cityscapes

Output: 1) qualitative visualizations on sample images, 2) quantitative mIoU for both.


## 1. Mount Drive and environment setup

In [16]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab_Projects/Anomaly-Segmentation
!git pull origin master 2>/dev/null || echo "(git pull skipped)"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab_Projects/Anomaly-Segmentation
Updating 2c01155..c9a8d86
(git pull skipped)


## 2. Install dependencies

We install directly in Colab's Python (no conda). Only the dependencies needed for the vendored EoMT inference, no `jsonargparse`, `gitignore_parser` or `wandb login`.

In [17]:
!pip install -q lightning timm transformers wandb pyyaml torchmetrics fvcore scipy pycocotools

## 3. Imports and path

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path("/content/drive/MyDrive/Colab_Projects/Anomaly-Segmentation")
EOMT_DIR  = REPO_ROOT / "eomt"
DATA_PATH = REPO_ROOT / "dataset" / "Validation_Dataset" / "Cityscapes"

# Repo root must be on sys.path so `utils.eval_semantic` resolves; eomt/ must be on
# sys.path so the YAML class_path entries (e.g. `datasets.cityscapes_semantic`) resolve.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(EOMT_DIR))

# Reusable helpers from the project module - same code path as the CLI in utils/eval_semantic.py.
from utils.eval_semantic import (
    build_model_and_data,
    infer_semantic_logits,
    infer_panoptic,
    build_coco_to_cs_lut,
    per_pixel_target,
    evaluate_semantic,
    print_iou_table,
    colorize_cs,
    panoptic_to_rgb,
    CS_CLASS_NAMES,
)
# Cityscapes data module from the vendored EoMT subtree.
from datasets.cityscapes_semantic import CityscapesSemantic

assert torch.cuda.is_available(), \
    "Enable GPU runtime: Runtime → Change runtime type → T4 GPU"
device = torch.device("cuda")
print("device:", device, "| torch:", torch.__version__)

## 4. Load the two checkpoints

We use `build_model_and_data` from [utils/eval_semantic.py](utils/eval_semantic.py): it reads the YAML config, instantiates encoder + network + Lightning module via `importlib`, and loads the `.bin` weights, same code path as the CLI. We pass `setup_data=False` so the data module returns only metadata (`num_classes`, `img_size`, `stuff_classes`) without trying to open the COCO zips.

In [ ]:
cs_config = EOMT_DIR / "configs" / "dinov2" / "cityscapes" / "semantic" / "eomt_base_640.yaml"
cs_ckpt   = REPO_ROOT / "models" / "eomt_cityscapes.bin"
print("Building Cityscapes-trained EoMT-")
cs_model, cs_meta = build_model_and_data(cs_config, cs_ckpt, DATA_PATH, device, setup_data=False)
print(f"  num_classes={cs_meta.num_classes}  img_size={cs_meta.img_size}")

In [ ]:
coco_config = EOMT_DIR / "configs" / "dinov2" / "coco" / "panoptic" / "eomt_base_640_2x.yaml"
coco_ckpt   = REPO_ROOT / "models" / "eomt_coco.bin"
print("Building COCO-trained EoMT-")
coco_model, coco_meta = build_model_and_data(coco_config, coco_ckpt, DATA_PATH, device, setup_data=False)
print(f"  num_classes={coco_meta.num_classes}  img_size={coco_meta.img_size}")

## 5. Load the Cityscapes val dataset

Read directly from the two zips: `leftImg8bit_trainvaltest.zip` and `gtFine_trainvaltest.zip`. `CityscapesSemantic` is imported from the vendored EoMT subtree ([eomt/datasets/cityscapes_semantic.py](eomt/datasets/cityscapes_semantic.py)). Change `img_idx` to explore different images.

In [ ]:
print("Loading Cityscapes val dataset (from zips)")
cs_data = CityscapesSemantic(
    path=str(DATA_PATH), batch_size=1, num_workers=0, check_empty_targets=False,
).setup()
val_dataset = cs_data.val_dataloader().dataset
print(f"  {len(val_dataset)} validation images")

img_idx = 10   # <-- switch for different images
img, target_dict = val_dataset[img_idx]
print(f"image {img_idx}: shape={tuple(img.shape)}  "
      f"GT classes={sorted(target_dict['labels'].tolist())}")

## 6. Semantic prediction of the CS-trained model

Three panels: image, prediction, ground truth. Colors come from `colorize_cs` (Cityscapes 19-class palette, with `ignore_index=255` mapped to black).

In [ ]:
logits  = infer_semantic_logits(cs_model, img, device)
pred_cs = logits.argmax(0).cpu().numpy()
target  = per_pixel_target(target_dict).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
axes[0].set_title("Image")
axes[1].imshow(colorize_cs(pred_cs))
axes[1].set_title("EoMT-CS prediction")
axes[2].imshow(colorize_cs(target))
axes[2].set_title("Ground Truth")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Panoptic prediction of the COCO-trained model

Panoptic output: each pixel has both a semantic class (out of the 133 from COCO) and an instance id. The RGB rendering (random hue per class + black borders between segments) is built by `panoptic_to_rgb`, the same helper used by the CLI.

In [ ]:
sem, inst = infer_panoptic(coco_model, img, device)
print(f"unique semantic classes in panoptic prediction: {np.unique(sem).tolist()}")

panoptic_rgb = panoptic_to_rgb(sem, inst, coco_meta.num_classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img.permute(1, 2, 0).cpu().numpy()); axes[0].set_title("Image")
axes[1].imshow(panoptic_rgb);                       axes[1].set_title("EoMT-COCO panoptic")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 8. COCO to Cityscapes: remapping for comparison

To compare the two networks on the same metric, a mapping from COCO continuous id to Cityscapes train_id is needed. The `COCO_TO_CS` dictionary and `build_coco_to_cs_lut` helper live in [utils/eval_semantic.py](utils/eval_semantic.py). Unmappable COCO classes (e.g., `pole`, `rider`) end up as `255` (ignore): in the mIoU, those pixels will not be counted. "Strict" approach: no free credit for what we cannot translate.

In [ ]:
logits_coco = infer_semantic_logits(coco_model, img, device)   # [133, H, W]
lut = build_coco_to_cs_lut(coco_meta.num_classes, device)
pred_coco_remap = lut[logits_coco.argmax(0)].cpu().numpy()     # {0..18, 255}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(img.permute(1,2,0).cpu().numpy());  axes[0].set_title("Image")
axes[1].imshow(colorize_cs(target));               axes[1].set_title("Ground Truth")
axes[2].imshow(colorize_cs(pred_cs));              axes[2].set_title("EoMT-CS")
axes[3].imshow(colorize_cs(pred_coco_remap));      axes[3].set_title("EoMT-COCO → CS remap")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 9. Quantitative: mIoU on Cityscapes val

Full loop on val (500 images) using `evaluate_semantic` from [utils/eval_semantic.py](utils/eval_semantic.py). For the COCO model we pass the LUT so predictions are remapped to Cityscapes train_ids before scoring (unmappable predictions are sent to ignore). For a quick sanity check set `MAX_IMAGES=50`; for the official number set it to `-1`.

In [ ]:
MAX_IMAGES = 50   # -1 for all 500 val images

print(f"--- EoMT-CS on Cityscapes val (max_images={MAX_IMAGES}) ---")
iou_cs = evaluate_semantic(cs_model, val_dataset, device, max_images=MAX_IMAGES, vis_prefix="cs")
print_iou_table("EoMT-CS", iou_cs)

In [ ]:
print(f"--- EoMT-COCO (remapped) on Cityscapes val (max_images={MAX_IMAGES}) ---")
lut = build_coco_to_cs_lut(coco_meta.num_classes, device)
iou_coco = evaluate_semantic(coco_model, val_dataset, device, lut=lut, max_images=MAX_IMAGES, vis_prefix="coco")
print_iou_table("EoMT-COCO (remapped)", iou_coco)

## 10. Final comparison

The EoMT-CS should clearly win: it was trained on the same 19 classes and performs argmax directly in that space. The COCO-trained model loses on classes without a COCO analog (pole, rider, traffic sign) and on classes where the remapping is approximate (terrain mapped from grass/dirt/mountain).

In [ ]:
print(f"{'class':<16} {'EoMT-CS':>9} {'EoMT-COCO':>11}")
print("-" * 40)
for c, a, b in zip(CS_CLASS_NAMES, iou_cs.tolist(), iou_coco.tolist()):
    print(f"{c:<16} {a*100:>8.2f}  {b*100:>10.2f}")
print("-" * 40)
print(f"{'mIoU':<16} {iou_cs[~torch.isnan(iou_cs)].mean().item()*100:>8.2f}  "
      f"{iou_coco[~torch.isnan(iou_coco)].mean().item()*100:>10.2f}")